# Regressão Linear Simples: Horas de Estudo × Nota da Prova

**Pergunta:** quem estuda mais horas tira notas maiores?

Modelo: $\text{Nota} = a + b \cdot \text{Horas de estudo}$

Este notebook refaz no Python a regressão feita no Excel e compara os resultados.

## 1. Importar bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

## 2. Carregar os dados
Rode a célula e selecione o arquivo `regressao_horas_estudo_x_nota.xlsx` no seu computador.

In [ ]:
from google.colab import files
uploaded = files.upload()
nome_arquivo = list(uploaded.keys())[0]

df = pd.read_excel(nome_arquivo, sheet_name='Dados')
df.head()

## 3. Análise exploratória

In [ ]:
print("Formato:", df.shape)
print("\nValores nulos:\n", df.isnull().sum())
df.describe().round(2)

### Qual variável tem mais relação com a nota?

In [ ]:
variaveis = ['Horas de estudo (x)', 'Horas de sono', 'Frequência (%)', 'Notas anteriores', 'Nota da prova (y)']
correl = df[variaveis].corr()['Nota da prova (y)'].drop('Nota da prova (y)').sort_values(ascending=False)
print(correl.round(3))

correl.plot(kind='barh', figsize=(7, 3), color='steelblue')
plt.title('Correlação de cada variável com a Nota da prova')
plt.xlabel('Correlação de Pearson (r)')
plt.grid(alpha=0.3, axis='x')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['Horas de estudo (x)'], df['Nota da prova (y)'], alpha=0.6)
plt.title('Horas de estudo × Nota da prova')
plt.xlabel('Horas de estudo')
plt.ylabel('Nota da prova')
plt.grid(alpha=0.3)
plt.show()

## 4. Regressão com todos os dados (mesmo método do Excel)

In [ ]:
x = df['Horas de estudo (x)']
y = df['Nota da prova (y)']

reg = stats.linregress(x, y)
a, b = reg.intercept, reg.slope
n = len(df)

y_prev = a + b * x
erro_padrao_estimativa = np.sqrt(((y - y_prev) ** 2).sum() / (n - 2))
t_stat = b / reg.stderr

print(f"Equação: Nota = {a:.2f} + {b:.3f} × Horas")
print(f"\nInclinação b:              {b:.4f}  (pontos por hora de estudo)")
print(f"Intercepto a:              {a:.4f}  (nota prevista com 0 h)")
print(f"Correlação (r):            {reg.rvalue:.4f}")
print(f"R²:                        {reg.rvalue**2:.4f}")
print(f"Erro padrão da estimativa: {erro_padrao_estimativa:.4f}")
print(f"Erro padrão de b:          {reg.stderr:.4f}")
print(f"Estatística t:             {t_stat:.2f}")
print(f"Valor-p:                   {reg.pvalue:.2e}")
print(f"n:                         {n}")

### Comparação: Excel × Colab

In [ ]:
comparacao = pd.DataFrame({
    'Excel': [1.634110, 23.618438, 0.776751, 0.603343, 4.286892, 17.354295],
    'Colab (Python)': [b, a, reg.rvalue, reg.rvalue**2, erro_padrao_estimativa, t_stat],
}, index=['Inclinação b', 'Intercepto a', 'Correlação r', 'R²', 'Erro padrão da estimativa', 'Estatística t'])
comparacao['Diferença'] = (comparacao['Colab (Python)'] - comparacao['Excel']).abs()
comparacao.round(4)

## 5. Reta de regressão

In [ ]:
x_linha = np.linspace(x.min(), x.max(), 100)

plt.figure(figsize=(8, 5))
plt.scatter(x, y, alpha=0.5, label='Alunos')
plt.plot(x_linha, a + b * x_linha, color='red', linewidth=2,
         label=f'Nota = {a:.2f} + {b:.3f} × Horas')
plt.title('Regressão Linear: Horas de estudo × Nota da prova')
plt.xlabel('Horas de estudo')
plt.ylabel('Nota da prova')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 6. Análise dos resíduos
Os resíduos devem ficar espalhados aleatoriamente em torno de zero.

In [ ]:
residuos = y - y_prev

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(y_prev, residuos, alpha=0.6)
ax[0].axhline(0, color='red', linestyle='--')
ax[0].set_title('Resíduos × Notas previstas')
ax[0].set_xlabel('Nota prevista')
ax[0].set_ylabel('Resíduo')

ax[1].hist(residuos, bins=20, edgecolor='black')
ax[1].set_title('Distribuição dos resíduos')
ax[1].set_xlabel('Resíduo')
plt.tight_layout()
plt.show()

## 7. Validação: treino (80%) e teste (20%)
Verifica se o modelo também acerta alunos que ele não viu no treino.

In [ ]:
X = df[['Horas de estudo (x)']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo = LinearRegression().fit(X_train, y_train)
y_pred = modelo.predict(X_test)

print(f"Equação (treino): Nota = {modelo.intercept_:.2f} + {modelo.coef_[0]:.3f} × Horas")
print(f"R² no teste: {r2_score(y_test, y_pred):.4f}")
print(f"MAE:  {mean_absolute_error(y_test, y_pred):.2f} pontos")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f} pontos")

## 8. Previsões

In [ ]:
horas = pd.DataFrame({'Horas de estudo': [2, 5, 8, 10, 12]})
horas['Nota prevista'] = (a + b * horas['Horas de estudo']).round(2)
horas

## 9. Conclusão

- **Equação:** Nota = 23,62 + 1,634 × Horas de estudo. Cada hora a mais de estudo soma cerca de **1,63 ponto** na nota.
- **r = 0,777 e R² = 0,603:** as horas de estudo explicam cerca de **60%** da variação das notas. O restante depende de outros fatores, como as notas anteriores (r = 0,43).
- **t = 17,35 e valor-p ≈ 0:** a relação é **positiva e estatisticamente significativa**.
- **Excel × Colab:** os coeficientes são iguais. As duas ferramentas chegam ao mesmo modelo.
- **Exemplo:** com 10 h de estudo, a nota prevista é **≈ 39,96**.